In [109]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, confusion_matrix
)

In [110]:
results_path = Path("../datasets/Manatees/Inference_results-2025-07-25_11-32-16-Manatees-SGC_Only-Manatee_train_val_splits.csv")  # <-- update
THRESHOLD = 0.5                                           # adjustable threshold
LABEL_COLUMN = "label"                                    # ground-truth column
# If your positive-class probability column is not "1", set it explicitly here:
POS_PROB_COLUMN = '1'  # e.g. "prob_call"; leave None to auto-detect
SPLIT_COLUMN = "split_type"     # column that marks dataset split
SPLIT_VALUE = "val"            # value to keep

In [111]:
df = pd.read_csv(results_path)

# Basic sanity checks
if SPLIT_COLUMN not in df.columns:
    raise ValueError(f"Expected split column '{SPLIT_COLUMN}' not found. "
                     f"Available columns: {list(df.columns)}")
if LABEL_COLUMN not in df.columns:
    raise ValueError(f"Expected ground-truth column '{LABEL_COLUMN}' not found. "
                     f"Available columns: {list(df.columns)}")

#Filter to test split
df = df[df[SPLIT_COLUMN].astype(str).str.lower() == SPLIT_VALUE.lower()].copy()
if df.empty:
    raise ValueError(f"No rows found with {SPLIT_COLUMN} == '{SPLIT_VALUE}'. Please verify the CSV.")
print(f"Loaded {len(df)} rows from split '{SPLIT_VALUE}'.")
# Try to auto-detect the positive probability column if not provided
if POS_PROB_COLUMN is None:
    candidate_cols = []
    # Common patterns: column named '1', columns starting with 'prob', etc.
    if '1' in df.columns:
        candidate_cols.append('1')
    candidate_cols.extend([c for c in df.columns if c.lower().startswith("prob")])
    candidate_cols = list(dict.fromkeys(candidate_cols))  # deduplicate
    
    if not candidate_cols:
        raise ValueError(
            "Could not auto-detect positive probability column. "
            "Set POS_PROB_COLUMN manually."
        )
    POS_PROB_COLUMN = candidate_cols[0]  # first match

C:\Users\amitg\AppData\Local\Temp\ipykernel_9364\3360669984.py:1: DtypeWarning: Columns (17,18,19,20,27,36,37) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(results_path)


Loaded 66322 rows from split 'val'.


In [112]:
df.head()

,Selection,View,Channel,begin_time,end_time,Low Freq (Hz),High Freq (Hz),Center Time (s),Delta Freq (Hz),Delta Time (s),...,StartMicInWater,EndMicInWater,filepath,>22kHz,MANATEE,label,call_length,split_type,0,1
0,2,Spectrogram 1,1,1032.106,1032.306,6827.2,12191.5,1049.429,5364.3,0.197,...,1031.939,4277.935,2017-Jul-08-12-67407878-RestingHole1A,NaN,NaN,0,0.2,val,0.997154,0.002846
1,2,Spectrogram 1,1,1032.306,1032.506,6827.2,12191.5,1049.429,5364.3,0.197,...,1031.939,4277.935,2017-Jul-08-12-67407878-RestingHole1A,NaN,NaN,0,0.2,val,0.983361,0.016639
2,2,Spectrogram 1,1,1032.506,1032.706,6827.2,12191.5,1049.429,5364.3,0.197,...,1031.939,4277.935,2017-Jul-08-12-67407878-RestingHole1A,NaN,NaN,0,0.2,val,0.981541,0.018459
3,2,Spectrogram 1,1,1032.706,1032.906,6827.2,12191.5,1049.429,5364.3,0.197,...,1031.939,4277.935,2017-Jul-08-12-67407878-RestingHole1A,NaN,NaN,0,0.2,val,0.996162,0.003838
4,2,Spectrogram 1,1,1032.906,1033.106,6827.2,12191.5,1049.429,5364.3,0.197,...,1031.939,4277.935,2017-Jul-08-12-67407878-RestingHole1A,NaN,NaN,0,0.2,val,0.981079,0.018921


In [113]:
def compute_metrics(dataframe, prob_col, threshold=0.5, label_col="label"):
    """
    Compute classification metrics at a given threshold.
    
    Args:
        dataframe: pandas DataFrame with label and probability columns.
        prob_col: column containing probability for the positive class.
        threshold: decision threshold to convert probabilities to class labels.
        label_col: ground truth column (0/1 or bool-like).
    """
    # Ground truth
    y_true = dataframe[label_col]
    # Ensure numeric 0/1
    y_true = y_true.astype(int)
    
    # Positive-class probabilities
    y_prob = dataframe[prob_col].astype(float)
    
    # Predictions at threshold
    y_pred = (y_prob >= threshold).astype(int)
    
    # Metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc_pr = average_precision_score(y_true, y_prob)  # threshold-independent
    
    return {
        "threshold": threshold,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "AUC-PR": auc_pr
    }

def print_confusion_matrix(dataframe, prob_col, threshold=0.5, label_col="label"):
    """
    Print confusion matrix at the given threshold.
    Rows = true labels, Columns = predicted labels.
    """
    y_true = dataframe[label_col].astype(int)
    y_prob = dataframe[prob_col].astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    cm_df = pd.DataFrame(cm,
                         index=["True 0", "True 1"],
                         columns=["Pred 0", "Pred 1"])
    print(f"Confusion Matrix (threshold={threshold}):")
    print(cm_df)
    print(f"\nTN={tn}  FP={fp}  FN={fn}  TP={tp}")

In [114]:
metrics = compute_metrics(df, prob_col=POS_PROB_COLUMN, threshold=THRESHOLD, label_col=LABEL_COLUMN)

print(f"Using probability column: {POS_PROB_COLUMN}")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

print_confusion_matrix(df, prob_col=POS_PROB_COLUMN, threshold=THRESHOLD, label_col=LABEL_COLUMN)

Using probability column: 1
threshold: 0.5000
accuracy: 0.9937
precision: 0.7362
recall: 0.6390
f1: 0.6842
AUC-PR: 0.6661
Confusion Matrix (threshold=0.5):
        Pred 0  Pred 1
True 0   65447     163
True 1     257     455

TN=65447  FP=163  FN=257  TP=455
